In [2]:
def load_screener_link(nse_symbol, bse_code):
    #Custom screener URL
    if nse_symbol != 'missing':
        company_url = f"https://www.screener.in/company/{nse_symbol}/"
    else:
        company_url = f"https://www.screener.in/company/{bse_code}/"
    return company_url

In [3]:
#PREPARE base list for discovery 
import pandas as pd
base_df = pd.read_csv('discovery_base_list.csv', dtype={'BSE Code': str})
base_df['Market Capitalization'] = base_df['Market Capitalization'].round(0).astype('Int64')
base_df = base_df.round(1)

base_df['BSE Code'] = base_df['BSE Code'].fillna('missing')
base_df['NSE Code'] = base_df['NSE Code'].fillna('missing')
base_df['stock_id'] = base_df['NSE Code'] + '_' + base_df['BSE Code'] 

import re
base_df.columns = [re.sub(r'\s+', '_', col.strip().lower()) for col in base_df.columns]
base_df.insert(0, 'stock_id', base_df.pop('stock_id'))
# Assuming you just ran the lowercase/underscore normalization:
base_df = base_df.rename(columns={
    'nse_code': 'nse_id',
    'bse_code': 'bse_id',
})
base_df.head(3)

# Create the 'screener_link' column
base_df['screener_link'] = base_df.apply(
    lambda row: load_screener_link(row['nse_id'], row['bse_id']), 
    axis=1
)

base_df.to_csv('dis_baselist_v1.csv', index=False)

In [12]:
import pandas as pd
df = pd.read_parquet('data_revenue_segments.parquet')
import numpy as np
df.loc[df['gics_conf_1'] <= 0.45, 'gics_pred_1'] = np.nan
df.loc[df['gics_conf_2'] <= 0.45, 'gics_pred_2'] = np.nan
df.loc[df['gics_conf_3'] <= 0.45, 'gics_pred_3'] = np.nan

In [13]:
# Select the columns you want to combine
cols_to_combine = ['gics_pred_1', 'gics_pred_2', 'gics_pred_3']

# Create the new list column
df['gics_categories'] = df[cols_to_combine].apply(
    lambda row: [val for val in row if pd.notna(val)], 
    axis=1
)

In [16]:
# List of columns you want to remove
cols_to_drop = [ 'gics_pred_1', 'gics_conf_1',
       'gics_pred_2', 'gics_conf_2', 'gics_pred_3', 'gics_conf_3',
       'gics_reasoning']

# Drop them and assign back to the dataframe
df = df.drop(columns=cols_to_drop)

In [19]:
df.to_parquet('dis_rev_segment_v1.parquet')

In [17]:
df.sample(10)

,stock_id,nse_id,bse_id,company_name,business_segment,segment_summary,status,commission_year,has_domestic_presence,has_international_presence,...,client_concentration,products_or_services,operational_locations,domestic_reach_states,international_markets,top_clients,effective_revenue,revenue_pct,yoy_revenue_change,gics_categories
4016,GSMFOILS_missing,GSMFOILS,missing,GSM FOILS LIMITED,Aluminum Pharma Foils,Manufactures specialized aluminum packaging so...,Established,2019,True,True,...,Diversified client base of over 90 customers; ...,"[Blister Foils, Aluminium Strip Pharma Foils, ...","[Vasai, Maharashtra, Ahmedabad, Gujarat]","[Pan-India, Maharashtra, Gujarat, Dadra & Naga...","[Bangladesh, Yemen, Vietnam, Europe, USA]","[Tier-1 Pharmaceutical Companies, Tier-2 Pharm...",133.80,100.00,227.70,"[Metal, Glass & Plastic Containers, Aluminum]"
4293,AKUMS_544222,AKUMS,544222,Akums Drugs & Pharmaceuticals Limited,International Branded Formulations,Exports proprietary pharmaceutical formulation...,Established,2015,False,True,...,Geographically diversified across more than 60...,"[Branded Tablets, Branded Capsules, Branded In...","[Mumbai, Haridwar]",[],"[Philippines, Uganda, Nigeria, Europe, South E...","[Global Distributors, Government Health Depart...",142.61,3.46,14.26,[Pharmaceuticals]
2926,MANALIPETC_500268,MANALIPETC,500268,Manali Petrochemicals Limited,Petrochemicals,"Manufactures Propylene Oxide, Propylene Glycol...",Established,1986,True,True,...,Highly diversified client base across multiple...,"[Propylene Oxide, Propylene Glycol (PG), Di Pr...","[Chennai, Tamil Nadu, Saykha, Gujarat, Singapo...",[Pan-India],"[United Kingdom, Singapore, Germany, Southeast...","[Pharmaceutical companies, Food processing com...",897.12,100.00,-13.10,"[Commodity Chemicals, Specialty Chemicals]"
5150,TATACONSUM_500800,TATACONSUM,500800,Tata Consumer Products Limited,Tata Starbucks (Joint Venture),Operates a network of premium coffee retail st...,Established,2012,True,False,...,Highly diversified retail consumer base.,"[Specialty Coffee, Teas, Refreshers, Sandwiche...","[Mumbai, New Delhi, Gurugram, Jaipur, Ludhiana...","[Maharashtra, Delhi, Haryana, Rajasthan, Punja...",[],[Retail Consumers],1277.00,6.73,5.00,[Restaurants]
4096,JWL_533272,JWL,533272,Jupiter Wagons Limited,Rail Mobility,"Manufactures freight wagons, CMS crossings, an...",Established,1979,True,True,...,Major customer is Indian Railways; partially m...,"[Open wagons, Covered wagons, Flat wagons, Def...","[Bandel, Kolkata, Jabalpur, Aurangabad, Khurdha]",[Pan-India],"[USA, Europe]","[Indian Railways, Ambuja Cement, ACC Limited, ...",3376.56,87.24,7.98,[Construction Machinery & Heavy Transportation...
4844,CESC_500084,CESC,500084,CESC Limited,Solar Cell and Module Manufacturing,Establishing a manufacturing ecosystem for TOP...,Just Launched,<NA>,True,False,...,Planned to serve internal captive demand and e...,"[TOPCon+ Solar Cells, Solar Modules]",[Greater Noida],[Uttar Pradesh],[],"[Internal Captive Demand, Renewable Energy Dev...",NaN,NaN,NaN,[Semiconductors]
4481,BALMLAWRIE_523319,BALMLAWRIE,523319,Balmer Lawrie & Co. Ltd.,Industrial Packaging,"Manufactures rigid industrial packaging, prima...",Established,1867,True,True,...,Broad portfolio of customers across diverse in...,"[Open-Head Drums, Tight-Head Drums, Plain Drum...","[Asaoti, Chennai, Chittoor, Navi Mumbai, Silva...",[Pan-India],[Global Markets],"[Major PSUs, MNCs, Local Oil Producers, Chemic...",831.05,35.23,0.79,"[Metal, Glass & Plastic Containers]"
702,missing_544091,missing,544091,Qualitek Labs Limited,"Testing, Inspection, and Certification (TIC) S...",Operates NABL-accredited laboratories providin...,Established,2018,True,True,...,Highly diversified client base serving major a...,"[Vehicle-level testing, EV and battery systems...","[Pune, Bhubaneshwar, Noida, Paradeep, Panchkul...","[Pan-India, Maharashtra, Odisha, Uttar Pradesh...","[Malaysia, Japan, Thailand, Netherlands, China...","[Tata Motors, Skoda-Volkswage

In [2]:
import pandas as pd
df = pd.read_parquet('dis_rev_segment_v1.parquet')
df.to_csv('rev_seg_fin_check.csv')